# Verify the bonus Triton kernel on a real CUDA GPU

`triton_kernels.py` implements a hand-written Triton kernel that fuses scale → additive-mask → row-softmax into a single kernel launch. It's a bonus/demonstration module — it is **not** called by the graded `UserOptimizedTransformer` path, so it has zero effect on correctness/performance grading either way.

It could not be runtime-tested during development: Triton kernels only execute on CUDA GPUs, and neither the CPU-only dev sandbox nor the Apple Silicon (MPS) machine used to validate the rest of this submission has one. This notebook exists so you don't have to take our word for it — it clones the repo and runs `tests/test_triton_kernel.py` directly, in about a minute.

**Before running:** `Runtime` → `Change runtime type` → select a `T4 GPU` (or any CUDA GPU). The free tier is enough for this.

In [ ]:
# Repo path for this submission
REPO_URL = "https://github.com/brdge77e/optimized-transformer-layer.git"

!git clone "$REPO_URL" repo
%cd repo

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
!pip install -q triton
import torch
print("CUDA available:", torch.cuda.is_available())
assert torch.cuda.is_available(), "Select a GPU runtime: Runtime > Change runtime type > T4 GPU"

In [ ]:
!python3 tests/test_triton_kernel.py

## What to look for

The script sweeps sequence lengths (16–2048), causal on/off, padding on/off, and dtypes (float32/float16/bfloat16), comparing the Triton kernel's output against plain PyTorch softmax at `rtol<0.02, atol<0.002` — the same tolerance the hackathon's own test cases use. Every row should print `PASS`. If everything passes, `reports/TECH_REPORT.md` §9's "reasoned-through, not proven" caveat can be upgraded to verified — feel free to note the GPU model and PyTorch/Triton versions you tested on.